# Qwen 3 — Basic Usage

## Imports

In [1]:
from pprint import pprint

import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("MPS (Apple Silicon GPU) available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

torch: 2.13.0
transformers: 5.14.1
MPS (Apple Silicon GPU) available: True
CUDA available: False


## Load Model and Tokenizer

In [2]:
MODEL_ID = "Qwen/Qwen3-1.7B"

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

print(f"Architecture: {model.config.architectures}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Architecture: ['Qwen3ForCausalLM']
Parameters: 1,720,574,976
Device: mps:0
Dtype: torch.bfloat16


## Single Turn Generation

In [9]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(chat)

<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
<think>

</think>




In [10]:
inputs = tokenizer(chat, return_tensors='pt').to(model.device)
pprint(inputs, sort_dicts=False, width=120)

{'input_ids': tensor([[151644,    872,    198,   3838,    374,    279,   6722,    315,   9625,
             30, 151645,    198, 151644,  77091,    198, 151667,    271, 151668,
            271]], device='mps:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='mps:0')}


In [11]:
outputs = model.generate(**inputs, max_new_tokens=128)
print(outputs)

tensor([[151644,    872,    198,   3838,    374,    279,   6722,    315,   9625,
             30, 151645,    198, 151644,  77091,    198, 151667,    271, 151668,
            271,    785,   6722,    315,   9625,    374,  12095,     13, 151645]],
       device='mps:0')


In [12]:
print(tokenizer.decode(outputs[0]))

<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
<think>

</think>

The capital of France is Paris.<|im_end|>


In [13]:
input_len = inputs["input_ids"].shape[-1]
response = tokenizer.decode(outputs[0][input_len:])
print(response)

The capital of France is Paris.<|im_end|>


In [14]:
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print(response)

The capital of France is Paris.


## System Prompt

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who responds in all capitals."},
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(chat)

<|im_start|>system
You are a helpful assistant who responds in all capitals.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
<think>

</think>




In [16]:
inputs = tokenizer(chat, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]
outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

THE CAPITAL OF FRANCE IS PARIS.


## Thinking Generation

In [18]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

print(chat)

<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



In [20]:
inputs = tokenizer(chat, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=1024)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

<think>
Okay, the user is asking for the capital of France. I know that France is a country in Europe, and I recall that Paris is the capital. But I need to make sure I'm correct. Let me think... Yes, Paris has been the capital of France for a long time. It's the largest city and a major cultural and關係 center. I should also mention that it's in the Île-de-France region. Wait, is there any chance the user might be confused with another country's capital? Like, sometimes people mix up capitals of different countries. But France's capital is definitely Paris. I should confirm that there's no other city that's the capital. Also, maybe mention the historical context, like when Paris was established as the capital. But the user probably just wants the direct answer. So the answer is Paris.
</think>

The capital of France is **Paris**. It is the largest city in France and serves as the political, economic, and cultural center of the country. Paris is renowned for landmarks like the Eiffel Tow

## Multi-Turn Generation

In [21]:
messages = [
    {"role": "user", "content": "What's the capital of France?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

The capital of France is Paris.


In [22]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of France is Paris.'}]


In [23]:
prompt = "What is a famous landmark there?"

messages.append({"role": "user", "content": prompt})
pprint(messages, sort_dicts=False)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of France is Paris.'},
 {'role': 'user', 'content': 'What is a famous landmark there?'}]


In [24]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

A famous landmark in Paris is the **Eiffel Tower**. It is one of the most iconic symbols of France and a popular tourist attraction. Another well-known landmark is the **Louvre Museum**, which houses many famous artworks, including the Mona Lisa and the Venus de Milo. The **Notre-Dame Cathedral** is also a major historical and architectural landmark in Paris.


## Streaming Generation

In [25]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
).to(model.device)

streamer = transformers.TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
outputs = model.generate(**inputs, max_new_tokens=128, streamer=streamer)

The capital of France is Paris.
